# Ⅰ第2回 部屋A 勾配を記録する

**目的**：学習の中で，重みがどれだけ直されているかを数字で見る．第1回では `loss.backward()` を 1 行書いただけで，計算された勾配は見えなかった．その勾配を取り出して層ごとに記録する．

**学べること**：勾配の大きさは層ごとに違う／学習が進むと勾配も変わる／勾配を見ると学習がうまく進んでいるかを診断できる（部屋B・C も同じ記録の仕方で調べる）．

**問い**：学習中，どの層の勾配が大きく，どう変わっていくか．

題材は第1回 GW3 試行①（FashionMNIST × MLP）．学習ループに **記録の 1 行** を足し，層ごとの勾配の大きさを折れ線グラフにする．
実行時間の目安は 5 エポックで 1〜2 分．

## (0) グループと役割の設定

In [ ]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 1                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) ライブラリのインポートと使用デバイス

In [ ]:
import math
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from common.device import get_device, synchronize
from common.fashion import loaders            # GW3 試行①と同じデータ（訓練 48,000 枚・検証 12,000 枚）
from common.logger import ResultLogger
from common.todo import todo_check, quiz_check
plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]   # Mac の日本語フォント（文字化け防止）

device = get_device()
print("使用デバイス:", device)

## (確認) 実験の前に：用語の確認

授業ページの「勾配と学習率」を読んでから答える．**全問正解すると実験に進める**（ちがうとヒントが出る）．

**Q1**　勾配とは何か
- ア 損失の値そのもの
- イ 損失を小さくするには，重みをどちらへ，どれだけ動かせばよいかを表す数
- ウ 正解率がどれだけ上がったか

**Q2**　勾配が計算されるのは学習ループのどの行か
- ア outputs = model(images)
- イ loss.backward()
- ウ optimizer.step()

**Q3**　層ごとの勾配の大きさを 1 つの数にする式は
- ア layer.weight.grad.norm()
- イ layer.weight.sum()
- ウ loss.item()

In [ ]:
# ▼ 下の " " の中に，ア・イ・ウのどれかを書いて実行する
Q1 = ""
Q2 = ""
Q3 = ""
quiz_check({    # 2 つ目の値は答え合わせ用の暗号（答えは書いていない）
    "Q1": (Q1, "86c731cb", "勾配は，重みを直す向きと量を表す．損失そのものではない"),
    "Q2": (Q2, "af2c0979", "backward で勾配を計算し，step でその勾配を使って重みを直す"),
    "Q3": (Q3, "ba2dd3cd", "勾配の全要素の 2 乗和の平方根（L2 ノルム）．重みそのものではなく，勾配（.grad）を使う"),
})

## (2) データ（GW3 試行①と同じ）

In [ ]:
torch.manual_seed(0)
train_loader, val_loader = loaders(batch_size=64)
images, labels = next(iter(train_loader))
print("ミニバッチの形:", tuple(images.shape), "| 1 エポックのステップ数:", len(train_loader))

## (3) モデル・損失関数・最適化手法（GW3 試行①と同じ）

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 512), nn.ReLU(),     # 層1（入力側）
    nn.Linear(512, 256), nn.ReLU(),         # 層2
    nn.Linear(256, 10),                     # 層3（出力側）
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model

## (4) 勾配を記録する仕組み

`loss.backward()` を実行すると，各層の重み `layer.weight` の横に，同じ形の勾配 `layer.weight.grad` ができる．
勾配は「損失を小さくするには，この重みをどちらへどれだけ動かせばよいか」を表す．
ここでは層ごとに勾配の **大きさ**（全要素の 2 乗和の平方根 ＝ L2 ノルム）を 1 つの数にして記録する．

In [ ]:
class GradNormRecorder:
    def __init__(self, model):
        # 全結合層（nn.Linear）の重みだけを，入力に近い順に「層1, 層2, …」と呼ぶ
        self.layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
        self.names = [f"層{i + 1}" for i in range(len(self.layers))]
        self.history = {name: [] for name in self.names}

    def record(self):
        for name, layer in zip(self.names, self.layers):
            # TODO 1: 「...」を消して，layer.weight.grad の大きさを 1 つの数（Python の float）にする式を書く
            #   ヒント：テンソルの L2 ノルムは .norm()，1 要素のテンソルを数にするのは .item()
            self.history[name].append(...)

    def to_frame(self):
        return pd.DataFrame(self.history)       # 行 = ステップ，列 = 層

todo_check("TODO 1")
print(GradNormRecorder(model).names)

## (5) 学習ループ（GW3 の⑩）に記録の 1 行を足す

`loss.backward()` で勾配ができ，`optimizer.step()` で重みが動く．記録するのはその **間**．

In [ ]:
def validate(model):
    """検証データの損失と正解率（GW3 の⑪と同じ）"""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

EPOCHS = 5
todo_check("TODO 2")
recorder = GradNormRecorder(model)
train_losses, val_log = [], []

synchronize(device)
t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        # TODO 2: この下の「...」を消して，recorder で勾配の大きさを記録する 1 行を書く
        ...
        optimizer.step()
        train_losses.append(loss.item())
    val_loss, val_acc = validate(model)
    val_log.append((val_loss, val_acc))
    print(f"エポック {epoch + 1}/{EPOCHS}  訓練損失 {sum(train_losses[-len(train_loader):]) / len(train_loader):.4f}"
          f"  検証損失 {val_loss:.4f}  検証正解率 {val_acc * 100:.2f}%")
synchronize(device)
train_sec = time.perf_counter() - t0
grads = recorder.to_frame()
print(f"学習時間 {train_sec:.1f} 秒 | 記録したステップ数 {len(grads)}")

## (6) グラフ：層ごとの勾配と損失

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
grads.rolling(50, min_periods=1).mean().plot(ax=axes[0], logy=True)
axes[0].set_title("層ごとの勾配の大きさ（50 ステップの移動平均）")
axes[0].set_xlabel("ステップ"); axes[0].set_ylabel("勾配の大きさ（対数目盛）")
pd.Series(train_losses).rolling(50, min_periods=1).mean().plot(ax=axes[1], color="black")
axes[1].set_title("訓練損失（50 ステップの移動平均）"); axes[1].set_xlabel("ステップ")
plt.tight_layout(); plt.show()

summary = pd.DataFrame({"最初の100ステップ": grads.head(100).mean(), "最後の100ステップ": grads.tail(100).mean()})
summary["入力側の層1との比"] = summary["最初の100ステップ"] / summary["最初の100ステップ"].iloc[0]
summary.round(4)

## (7) 記録

In [ ]:
metrics = {"val_loss": val_log[-1][0], "val_acc": val_log[-1][1], "train_sec": train_sec,
           "grad_start_first": grads.iloc[:100, 0].mean(), "grad_start_last": grads.iloc[:100, -1].mean(),
           "grad_end_first": grads.iloc[-100:, 0].mean(), "grad_end_last": grads.iloc[-100:, -1].mean()}
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d2", exercise="room_a", device=device)
logger.log_many(metrics, condition="relu_depth2_lr0.001", seed=0)
print("書き込み先:", logger.path)

## (8) 報告（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 1〜2 文）．全体共有ではこの 3 つを 2 分で話す．

1. 勾配が最も大きい層はどこか．入力側と出力側で何倍違うか：＿＿
2. 学習が進むと勾配はどう変わったか．訓練損失のグラフと形は似ていたか：＿＿
3. 余裕があれば (3) の `lr=0.001` を 0.01 に変えて (3)〜(6) を再実行する．勾配と損失のグラフはどう変わったか：＿＿